In [33]:
import os
import email
import email.policy
import numpy as np
from sklearn.model_selection import train_test_split
import re
from html import unescape
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.sparse import csr_matrix
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import precision_score, recall_score
from collections import Counter
SPAM_DIR = 'spam'
HAM_DIR = 'easy_ham'

In [34]:
ham_filenames = [name for name in sorted(os.listdir(HAM_DIR)) if len(name) > 20]
spam_filenames = [name for name in sorted(os.listdir(SPAM_DIR)) if len(name) > 20]

In [35]:
def load_email(isSpam, filename):
    directory = "spam" if isSpam else "easy_ham"
    with open(os.path.join(directory, filename), "rb") as file:
        return email.parser.BytesParser(policy=email.policy.default).parse(file)

In [36]:
ham_emails = [load_email(isSpam=False, filename=curName) for curName in ham_filenames]
spam_emails = [load_email(isSpam=True, filename=curName) for curName in spam_filenames]

In [37]:
print(ham_emails[1].get_content().strip())

Martin A posted:
Tassos Papadopoulos, the Greek sculptor behind the plan, judged that the
 limestone of Mount Kerdylio, 70 miles east of Salonika and not far from the
 Mount Athos monastic community, was ideal for the patriotic sculpture. 
 
 As well as Alexander's granite features, 240 ft high and 170 ft wide, a
 museum, a restored amphitheatre and car park for admiring crowds are
planned
---------------------
So is this mountain limestone or granite?
If it's limestone, it'll weather pretty fast.

------------------------ Yahoo! Groups Sponsor ---------------------~-->
4 DVDs Free +s&p Join Now
http://us.click.yahoo.com/pt6YBB/NXiEAA/mG3HAA/7gSolB/TM
---------------------------------------------------------------------~->

To unsubscribe from this group, send an email to:
forteana-unsubscribe@egroups.com

 

Your use of Yahoo! Groups is subject to http://docs.yahoo.com/info/terms/


In [38]:
for header, value in spam_emails[0].items():
    print(header,":",value)

Return-Path : <12a1mailbot1@web.de>
Delivered-To : zzzz@localhost.spamassassin.taint.org
Received : from localhost (localhost [127.0.0.1])	by phobos.labs.spamassassin.taint.org (Postfix) with ESMTP id 136B943C32	for <zzzz@localhost>; Thu, 22 Aug 2002 08:17:21 -0400 (EDT)
Received : from mail.webnote.net [193.120.211.219]	by localhost with POP3 (fetchmail-5.9.0)	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 13:17:21 +0100 (IST)
Received : from dd_it7 ([210.97.77.167])	by webnote.net (8.9.3/8.9.3) with ESMTP id NAA04623	for <zzzz@spamassassin.taint.org>; Thu, 22 Aug 2002 13:09:41 +0100
From : 12a1mailbot1@web.de
Received : from r-smtp.korea.com - 203.122.2.197 by dd_it7  with Microsoft SMTPSVC(5.5.1775.675.6);	 Sat, 24 Aug 2002 09:42:10 +0900
To : dcek1a1@netsgo.com
Subject : Life Insurance - Why Pay More?
Date : Wed, 21 Aug 2002 20:31:57 -1600
MIME-Version : 1.0
Message-ID : <0103c1042001882DD_IT7@dd_it7>
Content-Type : text/html; charset="iso-8859-1"
Content-Transfer-Encoding : qu

In [39]:
# Подготовка данных

In [40]:
x = ham_emails + spam_emails
y = [0] * len(ham_emails) + [1] * len(spam_emails)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.25, random_state=50)

In [41]:
# Преобразуем html в простой текст
def html_to_plain_text(html):
    text = re.sub('<head.*?>.*?</head>', '', html, flags=re.M | re.S | re.I)
    text = re.sub(r'<a\s.*?>', ' HYPERLINK ', text, flags=re.M | re.S | re.I)
    text = re.sub('<.*?>', '', text, flags=re.M | re.S)
    text = re.sub(r'(\s*\n)+', '\n', text, flags=re.M | re.S)
    return unescape(text)

In [42]:
# Преобразование писем, в которых есть html
def email_to_text(email):
    html = None
    for part in email.walk():
        ctype = part.get_content_type()
        if not ctype in ("text/plain", "text/html"):
            continue
        try:
            content = part.get_content()
        except:
            content = str(part.get_payload())
        if ctype == "text/plain":
            return content
        else:
            html = content
    if html:
        return html_to_plain_text(html)

In [43]:
for email, label in zip(x_train, y_train):
    if label == 1:
        print(email_to_text(email)[:100]) 

TIRED OF THE BULL OUT THERE?
Want To Stop Losing Money?

WANT A REAL MONEY MAKER?
RECEIVE $1,000-$5,
1) Fight The Risk of Cancer!
http://www.adclick.ws/p.cfm?o=315&s=pk007

2) Slim Down - Guaranteed to
URGENT PRIVATE & EXTREMELY CONFIDENTIAL



Dear ,

With profound interest and in utmost confidence, 
Hi -

( http://club.4tfox.com )

You get emails every day, offering to show you how to make money.
M

ATTENTION: This is a MUST for ALL Computer Users!!!
*NEW-Special Package Deal!*
Norton SystemWorks 

    BOOST
        Your Reception
        on
        any cell phone or cordless
    			 
           
Dear friend,

I am Mrs. Sese-seko widow of late President Mobutu Sese-seko of Zaire, now known as De
Do You Want To Teach and Grow Rich?





If you are a motivated and qualified communicator, I will p

Congratulations JM@NETNOTEINC.COM!
Dear JM@NETNOTEINC.COM,
You've been selected
to receive a bottle

   HYPERLINK
Tremendous Savings
on Toners, 
Inkjets, FAX, and Thermal Replenishables!!
 HY

TypeError: 'NoneType' object is not subscriptable

In [44]:
# Преобразователь письма в словарь(слово-количество)
class EmailToWordCounterTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, strip_headers=True, lower_case=True, remove_punctuation=True,
                 replace_urls=True, replace_numbers=True, stemming=True):
        self.strip_headers = strip_headers
        self.lower_case = lower_case
        self.remove_punctuation = remove_punctuation
        self.replace_urls = replace_urls
        self.replace_numbers = replace_numbers
        self.stemming = stemming
    def fit(self, x, y=None):
        return self
    def transform(self, x, y=None):
        x_transformed = []
        for email in x:
            text = email_to_text(email) or ""
            if self.lower_case:
                text = text.lower()
            if self.replace_urls and url_extractor is not None:
                urls = list(set(url_extractor.find_urls(text)))
                urls.sort(key=lambda url: len(url), reverse=True)
                for url in urls:
                    text = text.replace(url, " URL ")
            if self.replace_numbers:
                text = re.sub(r'\d+(?:\.\d*(?:[eE]\d+))?', 'NUMBER', text)
            if self.remove_punctuation:
                text = re.sub(r'\W+', ' ', text, flags=re.M)
            word_counts = Counter(text.split())
            if self.stemming and stemmer is not None:
                stemmed_word_counts = Counter()
                for word, count in word_counts.items():
                    stemmed_word = stemmer.stem(word)
                    stemmed_word_counts[stemmed_word] += count
                word_counts = stemmed_word_counts
            x_transformed.append(word_counts)
        return np.array(x_transformed)

In [48]:
try:
    import nltk

    stemmer = nltk.PorterStemmer()
    for word in ("Computations", "Computation", "Computing", "Computed", "Compute", "Compulsive"):
        print(word, "=>", stemmer.stem(word))
except ImportError:
    print("Error: stemming requires the NLTK module.")
    stemmer = None

Computations => comput
Computation => comput
Computing => comput
Computed => comput
Compute => comput
Compulsive => compuls


In [49]:
try:
    import urlextract # may require an Internet connection to download root domain names
    
    url_extractor = urlextract.URLExtract()
    print(url_extractor.find_urls("Will it detect github.com and https://youtu.be/7Pq-S557XQU?t=3m32s"))
except ImportError:
    print("Error: replacing URLs requires the urlextract module.")
    url_extractor = None

['github.com', 'https://youtu.be/7Pq-S557XQU?t=3m32s']


In [50]:
x_few = x_train[:3]
x_few_wordcounts = EmailToWordCounterTransformer().fit_transform(x_few)
x_few_wordcounts

array([Counter({'number': 11, 'to': 10, 'the': 10, 'at': 8, 'numberbp': 5, 'i': 5, 'modem': 5, 'connect': 4, 'linux': 4, 'is': 4, 'thi': 4, 'speed': 4, 'but': 4, 'handshak': 4, 'snumber': 4, 'a': 3, 'set': 3, 'of': 3, 'in': 2, 'usr': 2, 'which': 2, 'remot': 2, 'fail': 2, 'work': 2, 'for': 2, 'data': 2, 'am': 2, 'disabl': 2, 'v': 2, 'just': 2, 'colin': 2, 'dublin': 2, 'ie': 2, 'hi': 1, 'all': 1, 'anyon': 1, 'ever': 1, 'tri': 1, 've': 1, 'got': 1, 'numberk': 1, 'faxmodem': 1, 'meant': 1, 'anoth': 1, 'same': 1, 'and': 1, 'have': 1, 'due': 1, 'nt': 1, 'port': 1, 'on': 1, 'side': 1, 'alway': 1, 'slightli': 1, 'higher': 1, 'that': 1, 'no': 1, 'good': 1, 'tx': 1, 'rx': 1, 'ing': 1, 'server': 1, 'as': 1, 'it': 1, 'insist': 1, 'talk': 1, 'treacl': 1, 'note': 1, 'minicom': 1, 'hyperterm': 1, 'window': 1, 'first': 1, 'time': 1, 'ani': 1, 'idea': 1, 'baud': 1, 'bit': 1, 'even': 1, 'pariti': 1, 'do': 1, 'atznumb': 1, 'reset': 1, 'then': 1, 'send': 1, 'init': 1, 'string': 1, 'fnumberenumbervnumberqn

In [51]:
# Преобразователь построения словаря часто повторяющихся слов
class WordCounterToVectorTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, vocabulary_size=1000):
        self.vocabulary_size = vocabulary_size
    def fit(self, X, y=None):
        total_count = Counter()
        for word_count in X:
            for word, count in word_count.items():
                total_count[word] += min(count, 10)
        most_common = total_count.most_common()[:self.vocabulary_size]
        self.most_common_ = most_common
        self.vocabulary_ = {word: index + 1 for index, (word, count) in enumerate(most_common)}
        return self
    def transform(self, X, y=None):
        rows = []
        cols = []
        data = []
        for row, word_count in enumerate(X):
            for word, count in word_count.items():
                rows.append(row)
                cols.append(self.vocabulary_.get(word, 0))
                data.append(count)
        return csr_matrix((data, (rows, cols)), shape=(len(X), self.vocabulary_size + 1))

In [55]:
vocab_transformer = WordCounterToVectorTransformer(vocabulary_size=10000)
x_few_vectors = vocab_transformer.fit_transform(x_few_wordcounts)
x_few_vectors

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 213 stored elements and shape (3, 10001)>

In [56]:
x_few_vectors.toarray()

array([[ 0, 11, 10, ...,  0,  0,  0],
       [ 0,  0,  3, ...,  0,  0,  0],
       [ 0, 10,  4, ...,  0,  0,  0]], shape=(3, 10001))

In [139]:
vocab_transformer.vocabulary_

{'number': 1,
 'to': 2,
 'the': 3,
 'at': 4,
 'i': 5,
 'numberbp': 6,
 'modem': 7,
 'but': 8,
 'url': 9,
 'list': 10,
 'connect': 11,
 'linux': 12,
 'a': 13,
 'is': 14,
 'thi': 15,
 'speed': 16,
 'handshak': 17,
 'of': 18,
 'snumber': 19,
 'cv': 20,
 'set': 21,
 'on': 22,
 'for': 23,
 'use': 24,
 'in': 25,
 'usr': 26,
 'which': 27,
 'and': 28,
 'have': 29,
 'remot': 30,
 'fail': 31,
 'work': 32,
 'data': 33,
 'server': 34,
 'as': 35,
 'it': 36,
 'time': 37,
 'ani': 38,
 'am': 39,
 'disabl': 40,
 'v': 41,
 'just': 42,
 'instead': 43,
 's': 44,
 'colin': 45,
 'dublin': 46,
 'ie': 47,
 'm': 48,
 'sf': 49,
 'write': 50,
 'repositori': 51,
 'add': 52,
 'css': 53,
 'how': 54,
 'hi': 55,
 'all': 56,
 'anyon': 57,
 'ever': 58,
 'tri': 59,
 've': 60,
 'got': 61,
 'numberk': 62,
 'faxmodem': 63,
 'meant': 64,
 'anoth': 65,
 'same': 66,
 'due': 67,
 'nt': 68,
 'port': 69,
 'side': 70,
 'alway': 71,
 'slightli': 72,
 'higher': 73,
 'that': 74,
 'no': 75,
 'good': 76,
 'tx': 77,
 'rx': 78,
 'ing': 

In [58]:
preprocess_pipeline = Pipeline([
    ("email_to_wordcount", EmailToWordCounterTransformer()),
    ("wordcount_to_vector", WordCounterToVectorTransformer()),
])

x_train_transformed = preprocess_pipeline.fit_transform(x_train)

In [59]:
log_clf = LogisticRegression(solver="liblinear", random_state=42)
score = cross_val_score(log_clf, x_train_transformed, y_train, cv=3, verbose=3)
score.mean()

[CV] END ................................ score: (test=0.980) total time=   0.0s
[CV] END ................................ score: (test=0.984) total time=   0.1s
[CV] END ................................ score: (test=0.991) total time=   0.0s


np.float64(0.9848888888888889)

In [60]:
x_test_transformed = preprocess_pipeline.transform(x_test)

log_clf = LogisticRegression(solver="liblinear", random_state=42)
log_clf.fit(x_train_transformed, y_train)

y_pred = log_clf.predict(x_test_transformed)

In [61]:
precision_score(y_test, y_pred)

0.959349593495935

In [62]:
recall_score(y_test, y_pred)

0.9516129032258065

In [63]:
log_clf.predict(x_test_transformed[0])

array([0])

In [64]:
y_test[0]

0

In [65]:
log_clf.predict(x_test_transformed[100])

array([0])

In [66]:
y_test[100]

0

In [67]:
log_clf.predict(x_test_transformed[749])

array([1])

In [68]:
y_test[749]

1

In [140]:
import pandas as pd
from email import message_from_string
data = {
    'text': [
        "Congratulations! You've Won a Prize! You are one of our lucky winners! We are thrilled to inform you that you have won $1,000,000 in our sweepstakes!",
        "Hello John, I hope you are doing well. Just wanted to check in and see if you received my last email.",
        "Thank you for email!"
    ],
    'label': [
        '1', 
        '0',
        '0'
    ]
}

In [141]:
test_data = pd.DataFrame(data)
test_data.to_csv('spam_test_data.csv', index=False)

In [142]:
test_data = pd.read_csv('spam_test_data.csv')

In [143]:
x_test2 = test_data['text']
y_test2 = test_data['label']

In [144]:
email_objects = [message_from_string(email) for email in x_test2]

In [145]:
x_test_transformed2 = preprocess_pipeline.transform(email_objects)

In [146]:
log_clf.predict(x_test_transformed2)

array([1, 1, 0])

In [147]:
y_test2

0    1
1    0
2    0
Name: label, dtype: int64